#  1 — Install Dependencies

In [ ]:
!pip install -q transformers sentencepiece




#  2 — Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5EncoderModel
from tqdm import tqdm



#  3 — Paths

In [ ]:
DATA_PATH = "/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language"

train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
test_df = pd.read_csv(f"{DATA_PATH}/test.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)


# 4 — Token Parser

In [ ]:
def parse_tokens(s, seq_len=120):
    if pd.isna(s):
        return np.zeros(seq_len, dtype=int)

    arr = np.array(list(map(int, s.split())))

    if len(arr) > seq_len:
        arr = arr[:seq_len]
    else:
        arr = np.pad(arr, (0, seq_len - len(arr)))

    return arr



#  5 — Dataset Loader

In [ ]:
class MotionDataset(Dataset):
    def __init__(self, df, tokenizer, seq_len=120):
        self.df = df
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        tokens = self.tokenizer(
            str(row["gloss"]),
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        motion = parse_tokens(row["base_tokens"], self.seq_len)

        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "motion": torch.tensor(motion, dtype=torch.long)
        }


# 6 — Tokenizer & Loader

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

dataset = MotionDataset(train_df, tokenizer)

loader = DataLoader(dataset, batch_size=16, shuffle=True)


# 7 — Lightweight Model

In [ ]:
class LiteMotionGenerator(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()

        self.encoder = T5EncoderModel.from_pretrained("t5-small")

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.text_proj = nn.Linear(512, hidden)

        layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=4,
            dim_feedforward=hidden * 2,
            batch_first=True,
        )

        self.transformer = nn.TransformerEncoder(layer, num_layers=2)

        self.head = nn.Linear(hidden, 512)

    def forward(self, ids, mask, seq_len):
        with torch.no_grad():
            enc = self.encoder(ids, attention_mask=mask).last_hidden_state

        pooled = enc.mean(dim=1)

        x = self.text_proj(pooled)
        x = x.unsqueeze(1).repeat(1, seq_len, 1)

        x = self.transformer(x)

        logits = self.head(x)
        return logits





# 8 — Training Setup

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = LiteMotionGenerator().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(ignore_index=0)


# 9 — Training Loop

In [ ]:
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(loader):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        motion = batch["motion"].to(device)

        seq_len = motion.size(1)

        logits = model(ids, mask, seq_len)

        loss = criterion(
            logits.reshape(-1, 512),
            motion.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.2f}")


# 10 — Test Dataset

In [ ]:
class TestDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        tokens = tokenizer(
            str(row["gloss"]),
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )

        return {
            "id": int(row["id"]),
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
        }


In [ ]:
test_dataset = TestDataset(test_df, tokenizer)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)


# 11 — Inference

In [ ]:
outputs = []

model.eval()

with torch.no_grad():
    for batch in tqdm(test_loader):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)

        logits = model(ids, mask, 120)
        preds = logits.argmax(-1).cpu().numpy()

        for i in range(len(batch["id"])):
            sid = int(batch["id"][i])
            seq = preds[i]

            row = [sid]
            for _ in range(6):
                row.append(" ".join(map(str, seq)))

            outputs.append(row)


# 12 — Submission

In [ ]:
test_df = pd.read_csv(
    "/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/test.csv"
)

test_ids = test_df["id"].values


In [ ]:
submission = pd.DataFrame(outputs, columns=[
    "id",
    "base_tokens",
    "residual_1",
    "residual_2",
    "residual_3",
    "residual_4",
    "residual_5",
])

submission = submission.set_index("id")
submission = submission.loc[test_ids].reset_index()


In [ ]:
submission.to_csv("submission_final.csv", index=False)


In [ ]:
print(len(submission), len(test_ids))
print((submission["id"].values == test_ids).all())
